In [1]:
"""
This script is used to create the pycistopic object

authors: Roy Oelen, Martijn van der Werf

"""

'\nThis script is used to create the pycistopic object\n\nauthors: Roy Oelen, Martijn van der Werf\n\n'

In [14]:
# imports
from pycisTopic.cistopic_class import CistopicObject
from scipy.sparse import csr_matrix
from scipy.io import mmread
import gzip
from sklearn.preprocessing import binarize
import pandas as pd
import os
from pycisTopic.lda_models import run_cgs_models_mallet
from numpy.random import choice
from pycisTopic.lda_models import evaluate_models
import pickle
import numpy as np
import glob
import re
# md5 checksum creation
import hashlib

In [3]:
#######################################
# read the openness to nucleus matrix #
#######################################

# location of matrix
#fragment_matrix_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/matrix_merged.mtx.gz'
fragment_matrix_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_and_minor_celltypes/matrix_merged.mtx'
# open connection
#fragment_matrix_gzip = open(fragment_matrix_loc)
# read as coo matrix
#coo_fragment_matrix = mmread(fragment_matrix_gzip)
coo_fragment_matrix = mmread(fragment_matrix_loc)
# convert to csr format
fragment_matrix = csr_matrix(coo_fragment_matrix)
# close file handle
#fragment_matrix_gzip.close()

In [4]:
##########################################
# read nucleus barcodes and region names #
##########################################

# locations
barcodes_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_and_minor_celltypes/barcodes.tsv.gz'
features_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_and_minor_celltypes/features.tsv.gz'

# load region names
region_names_file = gzip.open(features_loc, 'rb')
region_names = pd.read_csv(region_names_file, sep='\t', header=None).iloc[:,0].to_list()
region_names = [i.replace('-', ':', 1) for i in region_names]

# Read cell names
cell_names_file = gzip.open(barcodes_loc, 'rb')
cell_names = pd.read_csv(cell_names_file, sep='\t', header=None).iloc[:,0].to_list()

In [5]:
#########################################
# set up path to all the fragment files #
#########################################

# fragment count cPeaks path
fragment_path = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/rounded_fragments/'
# List all files in the directory
fragment_files = os.listdir(fragment_path)
# compile a pattern
pattern = re.compile(r'.*tsv\.gz$')
# filter by pattern
fragment_files = [f for f in fragment_files if os.path.isfile(os.path.join(fragment_path, f)) and pattern.match(f)]
# order the files
fragment_files.sort()
# add the path
fragment_files = [os.path.join(fragment_path, f) for f in fragment_files]

In [6]:
#############################
# read the nucleus metadata #
#############################

# location of the metadata
cell_metadata_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_and_minor_celltypes/metadata.tsv.gz'
cell_metadata = pd.read_csv(cell_metadata_loc, sep = '\t', dtype = {'confined_condition': 'category', 'unconfined_condition': 'category','final_condition': 'category','LONG_COVID': 'category'})

# Subset cell metadata 
cell_metadata = cell_metadata[cell_metadata.bc.isin(cell_names)]
# Filter cell names
cell_names = cell_metadata.bc.to_list()
# Names as rownames 
cell_metadata.index = cell_metadata.bc

/local/2238705/ipykernel_135142/26829561.py:7: DtypeWarning: Columns (0,1,4,5,6,7,10,11,12,13,14,15,18,19,22,23,24,25,26,28,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_metadata = pd.read_csv(cell_metadata_loc, sep = '\t', dtype = {'confined_condition': 'category', 'unconfined_condition': 'category','final_condition': 'category','LONG_COVID': 'category'})


In [7]:
##############################
# read region inclusion list #
##############################

# location of the file with the regions we want to keep
region_inclusion_list_loc = '//groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_and_minor_celltypes/mo_pct0001_regions.txt.gz'
# read this file
region_inclusion_list = pd.read_csv(region_inclusion_list_loc, sep='\t', header=None).iloc[:,0].to_list()

In [8]:
##############################
# filter by region inclusion #
##############################

# filter the fragment matrix and the region names to be data that is in the inclusion list
#fragment_matrix = [fragment_matrix[i] for i in range(len(region_names)) if region_names[i] in region_inclusion_list]
#region_names = region_names = [name for name in region_names if name in region_inclusion_list]

# convert region_names to a NumPy array
region_names_array = np.array(region_names)
# create a boolean mask
region_mask = np.isin(region_names_array, region_inclusion_list)

# use the mask to filter the rows of CSR matrix
fragment_matrix = fragment_matrix[region_mask]
# filter region_names based on same mask
region_names_array = region_names_array[region_mask]
# convert back to list
region_names = region_names_array.tolist()

In [9]:
#####################################
# binarize the accessibility matrix #
#####################################

# binarization of accessibility matrix
fragment_matrix_binarized = binarize(fragment_matrix, threshold=0)

In [10]:
##########################################
# create metadata for all of the regions #
##########################################

# location of the cpeaks annotation
cpeaks_annotation_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/region_annotations.tsv.gz'
# read the annotations
cpeaks_annotation = pd.read_csv(cpeaks_annotation_loc, sep = '\t')
# add a name column
cpeaks_annotation['name'] = cpeaks_annotation['chr_hg38'].astype(str) + ':' + cpeaks_annotation['start_hg38'].astype(str) + '-' + cpeaks_annotation['end_hg38'].astype(str)

# Subset region metadata metadata 
cpeaks_annotation = cpeaks_annotation[cpeaks_annotation.name.isin(region_names)]
# Names as rownames 
cpeaks_annotation.index = cpeaks_annotation.name

In [11]:
#############################
# create pycistopic object #
#############################

# construct object
cistopic_obj = CistopicObject(
    fragment_matrix=fragment_matrix,
    binary_matrix=fragment_matrix_binarized,
    cell_names=cell_names,
    region_names=region_names,
    region_data=cpeaks_annotation,
    cell_data=cell_metadata,
    path_to_fragments=fragment_files,
    project='wijst_multiome'
)


In [12]:
##########################
# save pycistopic object #
##########################

# location to store the object
pycistopic_object_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/all_nuclei_and_regions_major_minor.pkl'

# save the object
with open(pycistopic_object_loc, 'wb') as f:
   pickle.dump(cistopic_obj, f)


In [15]:
##########################################
# create md5 checksum for created object #
##########################################

def create_md5_file(input_file):
    """
    Creates an MD5 hash of the specified file and writes it to a new file with the same name but .md5 added to the extension.

    Args:
        input_file (str): The path to the input file for which the MD5 hash should be created.

    Returns:
        int: Returns 0 on success, 1 on failure.

    Raises:
        FileNotFoundError: Thrown if the input file does not exist.
        IOError: Thrown if there is an error reading the input file (like permission denied) or writing the output md5 file.
    """
    try:
        # get an md5 of the file
        digest = None
        with open(input_file, "rb") as f:
            # try Python 3.11+ method if it is available
            if callable(getattr(hashlib, 'file_digest', None)):
                # digest with one command
                digest = hashlib.file_digest(f, 'md5')
            # or the older 3.8+ method if we don't have the newer method
            else:
                # initialize digest
                digest = hashlib.md5()
                # read file in chunks
                while chunk := f.read(8192):
                    # update digestion
                    digest.update(chunk)       
        # get the output path of the md5
        output_md5_loc = ''.join([input_file, '.md5'])
        # and write that
        with open(output_md5_loc, "w") as m:
            m.write(digest.hexdigest())
        # upon success, return 0
        return 0
    except Exception as e:
        print(f"Exception occured upon md5 file creation: {e}")
        return 1


# and make an md5
create_md5_file(pycistopic_object_loc)

0